# Домашние задани по предмету "Язык Python для разработчиков" 

Студент Фонов Александр Денисович

## Задание 1

In [ ]:
from datetime import datetime

from enum import Enum
    
class OperationType(Enum):
    WITHDRAW = "withdraw"
    DEPOSIT = "deposit"
    # Фиксируем проверку баланса в истории операций для аудита
    BALANCE_CHECK = "balance_check"


class OperationStatus(Enum):
    SUCCESS = "success"
    FAIL = "fail"


class Operation:

    type: OperationType
    amount: float
    datetime: datetime
    balance: float
    status: OperationStatus

    def __init__(
        self,
        type: OperationType,
        amount: float,
        datetime: datetime,
        balance: float,
        status: OperationStatus
    ) -> None:
        self.type = type
        self.amount = amount
        self.datetime = datetime
        self.balance = balance
        self.status = status


class Account:

    # Используем общий счётчик, чтобы каждому новому счёту присваивать уникальный номер
    _account_counter: int = 1000
    _balance: float

    holder: str
    account_number: str
    operations_history: list[Operation]

    def __init__(
        self,
        account_holder: str,
        balance: float = 0
    ) -> None:
        if balance < 0:
            raise ValueError(
                "начальный баланс не может быть отрицательным"
            )

        self.holder = account_holder

        Account._account_counter += 1
        self.account_number = f"ACC-{Account._account_counter:04d}"

        self._balance = balance
        self.operations_history = []

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(
                "сумма пополнения должна быть положительной"
            )

        self._balance += amount

        operation = Operation(
            type=OperationType.DEPOSIT,
            amount=amount,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS
        )
        self.operations_history.append(operation)

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError(
                "сумма снятия должна быть положительной"
            )

        if amount > self._balance:
            # Неуспешную операцию тоже сохраняем, чтобы история отражала
            # все попытки изменения счёта, а не только успешные.
            operation = Operation(
                type=OperationType.WITHDRAW,
                amount=amount,
                datetime=datetime.now(),
                balance=self._balance,
                status=OperationStatus.FAIL
            )
            self.operations_history.append(operation)

            return False

        self._balance -= amount

        operation = Operation(
            type=OperationType.WITHDRAW,
            amount=amount,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS
        )
        self.operations_history.append(operation)

        return True

    def get_balance(self) -> float:
        # Проверку баланса также записываем в историю, поскольку она является
        # отдельным событием, полезным для аудита операций со счётом
        operation = Operation(
            type=OperationType.BALANCE_CHECK,
            amount=0,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS
        )
        self.operations_history.append(operation)

        return self._balance

    def get_history(self) -> list[Operation]:
        return self.operations_history

## Задание 2

In [ ]:
#др. библитеки импортировали в предыдущем блоке
import re

# Завели enum для типа операции, типа аккаунта и статуса операции,
# чтобы убрать валидацию при создании класса Operation и искл. др. ошибки,
# связанные с типизацией
class OperationType(Enum):
    WITHDRAW = "withdraw"
    DEPOSIT = "deposit"
    BALANCE_CHECK = "balance_check"
    INTEREST = "interest"


class OperationStatus(Enum):
    SUCCESS = "success"
    FAIL = "fail"


class AccountType(Enum):
    SAVING = "saving"
    CHECKING = "checking"
    DEFAULT = "default"


class Operation:
    type: OperationType
    amount: float | None
    datetime: datetime
    balance: float
    status: OperationStatus
    account_type: AccountType
    

    def __init__(
        self,
        type: OperationType,
        amount: float | None,        # т.к. он имеет тип Any
        datetime: datetime,
        balance: float,
        status: OperationStatus,
        account_type: AccountType
    ) -> None:
        self.type = type
        self.amount = amount
        self.datetime = datetime
        self.balance = balance
        self.status = status
        self.account_type = account_type


class Account:
    _account_counter: int = 1000

    account_type: AccountType = AccountType.DEFAULT

    holder: str
    account_number: str
    _balance: float
    operations_history: list[Operation]

    def __init__(
        self,
        account_holder: str,
        balance: float = 0
    ) -> None:

        self._validate_holder(account_holder)

        if balance < 0:
            raise ValueError(
                "начальный баланс не может быть отрицательным"
            )

        self.holder = account_holder

        Account._account_counter += 1
        self.account_number = (
            f"ACC-{Account._account_counter:04d}"
        )

        self._balance = balance
        self.operations_history = []

    @staticmethod
    def _validate_holder(account_holder: str) -> None:
        pattern = r"^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$"

        if not re.fullmatch(pattern, account_holder):
            raise ValueError(
                "имя должно быть в формате "
                "'Имя Фамилия' с заглавных букв"
            )

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(
                "сумма пополнения должна быть положительной"
            )

        self._balance += amount

        operation = Operation(
            type=OperationType.DEPOSIT,
            amount=amount,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS,
            account_type=self.account_type
        )

        self.operations_history.append(operation)

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError(
                "сумма снятия должна быть положительной"
            )

        if amount > self._balance:
            operation = Operation(
                type=OperationType.WITHDRAW,
                amount=amount,
                datetime=datetime.now(),
                balance=self._balance,
                status=OperationStatus.FAIL,
                account_type=self.account_type
            )

            self.operations_history.append(operation)

            return False

        self._balance -= amount

        operation = Operation(
            type=OperationType.WITHDRAW,
            amount=amount,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS,
            account_type=self.account_type
        )

        self.operations_history.append(operation)

        return True

    def get_balance(self) -> float:
        operation = Operation(
            type=OperationType.BALANCE_CHECK,
            amount=None,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS,
            account_type=self.account_type
        )

        self.operations_history.append(operation)

        return self._balance

    def get_history(self) -> list[Operation]:
        return self.operations_history

    def get_large_operations(self, n: int) -> list[Operation]:
        if n <= 0:
            raise ValueError("n должно быть положительным")

        operations = [
            operation
            for operation in self.operations_history
            if operation.amount is not None
            and operation.status == OperationStatus.SUCCESS
        ]

        operations.sort(
            key=lambda operation: (
                operation.amount,
                operation.datetime
            ),
            reverse=True
        )

        return operations[:n]


class CheckingAccount(Account):
    # переопределяем тип аккаунта базового класса на CHECKING
    account_type: AccountType = AccountType.CHECKING


class SavingsAccount(Account):
    # переопределяем тип аккаунта базового класса на SAVINGG
    account_type: AccountType = AccountType.SAVING

    def apply_interest(self, rate: float) -> None:
        if rate < 0:
            raise ValueError(
                "процентная ставка не может быть отрицательной"
            )

        interest = self._balance * rate / 100

        self._balance += interest

        operation = Operation(
            type=OperationType.INTEREST,
            amount=interest,
            datetime=datetime.now(),
            balance=self._balance,
            status=OperationStatus.SUCCESS,
            account_type=self.account_type
        )

        self.operations_history.append(operation)

    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError(
                "сумма снятия должна быть положительной"
            )
            
        # Для сберегательного счёта действуют ограничения:
        # за одну операцию нельзя снять больше половины текущего баланса.
        if amount > self._balance * 0.5:
            operation = Operation(
                type=OperationType.WITHDRAW,
                amount=amount,
                datetime=datetime.now(),
                balance=self._balance,
                status=OperationStatus.FAIL,
                account_type=self.account_type
            )

            self.operations_history.append(operation)

            return False

        return super().withdraw(amount)